# Praktikum EDA — Topik Dalam Data Mining

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/__REPO_SLUG__/blob/main/notebooks/praktikum01_eda.ipynb)

**EF235161 Topik Dalam Data Mining (P) — S-2 Teknik Informatika, ITS**
Exploratory Data Analysis, Data Preprocessing & Experimental Validity · sesi 120 menit

---

## Isi identitas Anda

| | |
|---|---|
| **Nama** | _(isi)_ |
| **NRP** | _(isi)_ |
| **Username GitHub** | _(isi)_ |

---

## Cara menyimpan pekerjaan Anda

Bekerja di Colab: **File → Save a copy in GitHub**, pilih repositori praktikum Anda, isi pesan commit. Ulangi setiap kali Anda ingin menyimpan — itulah commit Anda.

Bekerja lokal: `git add`, `git commit`, `git push` seperti biasa.

---

## Aturan sesi

- Jalankan sel **berurutan dari atas**. Jangan melompat.
- Setiap kali menjalankan sel, tulis satu kalimat interpretasi di sel Markdown di bawahnya. Menjalankan kode tanpa interpretasi berarti belum mengerjakan praktikum.
- Dataset `customer_churn_eda.csv` **sengaja dibuat cacat**. Tugas Anda menemukan cacatnya lewat bukti, lalu mempertanggungjawabkan setiap keputusan.
- Temuan akhir dituliskan di `laporan/LAPORAN.md`.

> **Pesan utama:** EDA bukan kegiatan membuat grafik. EDA adalah proses mengumpulkan bukti untuk mengambil keputusan preprocessing dan menjaga validitas eksperimen.

## Blok 0 — Setup & reproducibility

In [ ]:
import sys, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import sklearn

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.set_option("display.max_columns", 50)

print("Python      :", sys.version.split()[0])
print("NumPy       :", np.__version__)
print("pandas      :", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("RANDOM_STATE:", RANDOM_STATE)

Sel berikut memuat dataset secara otomatis dengan tiga lapis fallback: file lokal di `data/`, lalu URL repositori, lalu dialog upload. Anda tidak perlu mengatur apa pun.

Bila Anda ingin bekerja langsung di dalam klon repositori Anda sendiri, jalankan dulu:

```python
!git clone https://github.com/__REPO_SLUG__.git
%cd __REPO_NAME__
```

In [ ]:
import os, io, urllib.request

DATA_LOCAL = "data/customer_churn_eda.csv"
DATA_LOCAL_ALT = "../data/customer_churn_eda.csv"
DATA_URL = "https://raw.githubusercontent.com/__REPO_SLUG__/main/data/customer_churn_eda.csv"

def muat_dataset():
    """Tiga lapis: file lokal -> URL repositori -> dialog upload Colab."""
    for p in (DATA_LOCAL, DATA_LOCAL_ALT, "customer_churn_eda.csv"):
        if os.path.exists(p):
            print(f"Sumber: file lokal '{p}'")
            return pd.read_csv(p)
    try:
        print(f"Sumber: {DATA_URL}")
        with urllib.request.urlopen(DATA_URL, timeout=20) as r:
            return pd.read_csv(io.BytesIO(r.read()))
    except Exception as e:
        print("Gagal memuat dari URL:", e)
        print("Silakan unggah customer_churn_eda.csv secara manual.")
        from google.colab import files
        files.upload()
        return pd.read_csv("customer_churn_eda.csv")

df = muat_dataset()
print("Ukuran dataset:", df.shape)
df.head()

## Blok 1 — Struktur dataset

Pertanyaan pertama peneliti: **satu baris di dataset ini merepresentasikan apa?**

In [ ]:
df.info()
print("\nJumlah baris  :", len(df))
print("Jumlah kolom  :", df.shape[1])
print("customer_id unik:", df["customer_id"].nunique())

**Interpretasi Anda:** _(jumlah baris vs jumlah ID unik — apa dugaan Anda?)_

## Blok 2 — Tipe fitur: *file type* vs *semantic type*

`premium_member` bertipe `int64`, tetapi maknanya kategori biner. Perbedaan inilah sumber kesalahan preprocessing paling umum.

In [ ]:
profil = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "n_unique": df.nunique(),
    "n_missing": df.isna().sum(),
    "pct_missing": (df.isna().mean() * 100).round(2),
})
profil

**Tugas kecil:** sebutkan kolom yang TIDAK boleh menjadi fitur input, beserta alasannya.

## Blok 3 — Missing values

Yang penting bukan berapa persen yang hilang, tetapi **mengapa** hilang (MCAR / MAR / MNAR).

In [ ]:
miss = (pd.DataFrame({"n_missing": df.isna().sum(),
                      "pct": (df.isna().mean()*100).round(2)})
          .query("n_missing > 0").sort_values("pct", ascending=False))
print(miss)

for c in miss.index:
    print(f"\n{c} — churn rate saat nilai ada vs missing:")
    print(df.groupby(df[c].isna())["churn"].mean().rename({False: "ada nilai", True: "missing"}).round(4))

**Catatan:** jangan `dropna()`. Imputasi dilakukan **di dalam pipeline, setelah split** (alasannya di Blok 9).

## Blok 4 — Duplikat & inkonsistensi kategori

In [ ]:
print("Duplikat baris penuh :", df.duplicated().sum())
print("Duplikat customer_id :", df["customer_id"].duplicated().sum())

for c in ["city", "segment", "channel"]:
    print(f"\n--- {c} ---")
    print(df[c].value_counts(dropna=False))

Perbaikan berikut **deterministik** (tidak belajar apa pun dari data), sehingga aman dilakukan sebelum split.
Bandingkan dengan imputasi median, yang dihitung dari data dan karenanya **harus** setelah split.

In [ ]:
df_clean = df.drop_duplicates().copy()
for c in ["city", "segment", "channel"]:
    df_clean[c] = df_clean[c].str.strip().str.title()

print("Ukuran setelah dedup:", df_clean.shape)
print(df_clean["city"].value_counts(dropna=False))

## Blok 5 — Distribusi target & statistik deskriptif

In [ ]:
print(df_clean["churn"].value_counts())
print(df_clean["churn"].value_counts(normalize=True).round(4))

num_cols = ["age","monthly_income","tenure_months","monthly_spend",
            "transactions_per_month","support_tickets","satisfaction_score"]
print("\n--- Skewness ---")
print(df_clean[num_cols].skew().sort_values(ascending=False).round(2))
df_clean[num_cols].describe().T

**Dua angka yang menentukan seluruh desain eksperimen:**
1. Imbalance kelas → accuracy dilarang jadi metrik utama; split harus `stratify`.
2. Skewness ekstrem pada fitur nominal Rupiah → log-transform sebelum scaling.

## Blok 6 — Outlier (aturan IQR)

Outlier **tidak otomatis dibuang**. Buang hanya bila ada alasan domain bahwa nilainya mustahil.

In [ ]:
for c in ["monthly_income", "monthly_spend", "tenure_months"]:
    q1, q3 = df_clean[c].quantile([0.25, 0.75]); iqr = q3 - q1
    lo, hi = q1 - 1.5*iqr, q3 + 1.5*iqr
    mask = (df_clean[c] < lo) | (df_clean[c] > hi)
    print(f"{c:22s} outlier: {mask.sum():4d} ({mask.mean()*100:.1f}%)  batas atas: {hi:,.0f}")

fig, ax = plt.subplots(1, 3, figsize=(14, 3.5))
for a, c in zip(ax, ["monthly_income", "monthly_spend", "tenure_months"]):
    a.boxplot(df_clean[c].dropna()); a.set_title(c)
plt.tight_layout(); plt.show()

## Blok 7 — Bivariate analysis terhadap target

EDA menghasilkan **asosiasi**, bukan **kausalitas**. Perhatikan pilihan kata saat menulis temuan.

In [ ]:
display(df_clean.groupby("churn")[num_cols].median().T.round(2))

for c in ["segment", "channel", "city", "premium_member"]:
    print(f"\n--- {c} ---")
    print(df_clean.groupby(c)["churn"].agg(churn_rate="mean", n="count").round(3))

Boleh ditulis: *"Segmen Basic menunjukkan churn rate lebih tinggi."*
Tidak boleh: *"Segmen Basic menyebabkan churn."*

## Blok 8 — Korelasi: menemukan *redundant feature* dan *leakage*

Bagian terpenting sesi ini. Cari nilai yang terlalu bagus untuk jadi kenyataan.

In [ ]:
kandidat = num_cols + ["premium_member", "annual_spend", "churn_next_month_confirmed", "churn"]
print(df_clean[kandidat].corr()["churn"].sort_values(ascending=False).round(4))

In [ ]:
print(pd.crosstab(df_clean["churn_next_month_confirmed"], df_clean["churn"]))
print("\nKorelasi monthly_spend vs annual_spend:",
      round(df_clean[["monthly_spend","annual_spend"]].corr().iloc[0,1], 6))
print("\nRasio annual_spend / monthly_spend:")
print((df_clean["annual_spend"] / df_clean["monthly_spend"]).describe().round(3))

**Dua temuan, dua jenis masalah:**

- `annual_spend` = `monthly_spend` x 12 → **redundant feature**: tidak menambah informasi, merusak interpretasi koefisien.
- `churn_next_month_confirmed` identik dengan target → **target leakage**: informasi masa depan yang tidak tersedia saat prediksi harus dibuat.

Uji deteksi leakage bukan statistik, melainkan pertanyaan:
> *"Pada saat prediksi harus dibuat, apakah nilai kolom ini sudah tersedia?"*

---
## CHECKPOINT — Tulis 5 temuan EDA Anda di sini

Format wajib setiap temuan: **Temuan → Bukti (angka/sel mana) → Implikasi terhadap preprocessing atau validitas.**

1. *Temuan:* ... *Bukti:* ... *Implikasi:* ...
2.
3.
4.
5.

---

## Blok 9 — Keputusan preprocessing & split

| Keputusan | Bukti dari EDA |
|---|---|
| Drop `customer_id` | identifier, n_unique = n_baris (Blok 2) |
| Drop `annual_spend` | redundant, korelasi 1.0 (Blok 8) |
| Drop `churn_next_month_confirmed` | target leakage (Blok 8) |
| Hapus 12 duplikat | Blok 4 |
| Normalisasi string kategori | Blok 4 |
| Imputasi median / modus | Blok 3 |
| Log1p pada fitur skewed | Blok 5 |
| `stratify=y` | Blok 5 |
| Metrik utama PR-AUC | Blok 5 |

In [ ]:
from sklearn.model_selection import train_test_split

DROP = ["customer_id", "annual_spend", "churn_next_month_confirmed"]
X = df_clean.drop(columns=DROP + ["churn"])
y = df_clean["churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

print("Train:", X_train.shape, "| churn rate:", round(y_train.mean(), 4))
print("Test :", X_test.shape,  "| churn rate:", round(y_test.mean(), 4))

> **Mulai titik ini test set dikunci.** Tidak dilihat, tidak dipakai memilih model. Dibuka satu kali di Blok 11b.

## Blok 10 — Preprocessing pipeline & baseline

Pipeline memaksa imputer dan scaler belajar **hanya dari fold training** di setiap lipatan CV. Ini alat penjaga validitas, bukan sekadar kerapian kode.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier

skewed   = ["monthly_income", "monthly_spend"]
numerik  = ["age", "tenure_months", "transactions_per_month",
            "support_tickets", "satisfaction_score", "premium_member"]
kategori = ["city", "segment", "channel"]

pipe_skewed = Pipeline([("imputer", SimpleImputer(strategy="median")),
                        ("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
                        ("scaler", StandardScaler())])
pipe_numerik = Pipeline([("imputer", SimpleImputer(strategy="median")),
                         ("scaler", StandardScaler())])
pipe_kategori = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                          ("ohe", OneHotEncoder(handle_unknown="ignore"))])

pre = ColumnTransformer([("skewed", pipe_skewed, skewed),
                         ("numerik", pipe_numerik, numerik),
                         ("kategori", pipe_kategori, kategori)])

model = Pipeline([("pre", pre),
                  ("clf", LogisticRegression(max_iter=1000, class_weight="balanced",
                                             random_state=RANDOM_STATE))])
model

In [ ]:
dummy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
print("Accuracy DummyClassifier di test set:", round(dummy.score(X_test, y_test), 4))

Ingat angka baseline ini. Model yang akurasinya di bawah angka ini kalah dari menebak kelas mayoritas.

## Blok 11 — Cross-validation pada training set

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {"roc_auc": "roc_auc", "pr_auc": "average_precision",
           "f1": "f1", "accuracy": "accuracy"}

hasil = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring)
for m in scoring:
    s = hasil["test_" + m]
    print(f"{m:10s}: {s.mean():.3f} +/- {s.std():.3f}")

Accuracy di bawah baseline dummy adalah **konsekuensi desain** `class_weight="balanced"`, bukan kegagalan model.
Selalu laporkan rata-rata **beserta** standar deviasinya.

### Blok 11b — Evaluasi final (test set dibuka satu kali)

In [ ]:
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             confusion_matrix, classification_report)

model.fit(X_train, y_train)
proba = model.predict_proba(X_test)[:, 1]
pred  = model.predict(X_test)

print("ROC-AUC :", round(roc_auc_score(y_test, proba), 3))
print("PR-AUC  :", round(average_precision_score(y_test, proba), 3))
print("\nConfusion matrix:\n", confusion_matrix(y_test, pred))
print("\n", classification_report(y_test, pred, digits=3))

## Blok 12 — Demonstrasi *experimental validity*: apa yang terjadi bila leakage ikut dilatih?

Sebelum menjalankan sel di bawah, **tebak dulu** berapa ROC-AUC-nya.

In [ ]:
X_leak = df_clean.drop(columns=["customer_id", "annual_spend", "churn"])
Xl_tr, Xl_te, yl_tr, yl_te = train_test_split(
    X_leak, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

pre_leak = ColumnTransformer([
    ("skewed", pipe_skewed, skewed),
    ("numerik", pipe_numerik, numerik + ["churn_next_month_confirmed"]),
    ("kategori", pipe_kategori, kategori)])
model_leak = Pipeline([("pre", pre_leak),
                       ("clf", LogisticRegression(max_iter=1000, class_weight="balanced",
                                                  random_state=RANDOM_STATE))])

model_leak.fit(Xl_tr, yl_tr)
print("ROC-AUC test dengan leakage :", round(roc_auc_score(yl_te, model_leak.predict_proba(Xl_te)[:, 1]), 4))

cv_leak = cross_validate(model_leak, Xl_tr, yl_tr, cv=cv, scoring={"roc_auc": "roc_auc"})
print("ROC-AUC CV  dengan leakage :", round(cv_leak["test_roc_auc"].mean(), 4))
print("Bandingkan dengan model jujur di Blok 11b.")

**Perhatikan:** cross-validation pun menunjukkan nilai sempurna.
CV melindungi dari *overfitting*, **bukan** dari *target leakage*. Satu-satunya pertahanan adalah memahami asal-usul setiap kolom.

> Bila model Anda tiba-tiba mendapat AUC ~0.99 pada data nyata, reaksi pertama yang benar adalah **mencari leakage**, bukan menulis paper.

## Blok 13 — Reproducibility log

In [ ]:
log = {
    "random_state": RANDOM_STATE,
    "n_raw": len(df),
    "n_after_dedup": len(df_clean),
    "dropped_features": DROP,
    "n_train": len(X_train), "n_test": len(X_test),
    "cv": "StratifiedKFold(5, shuffle=True)",
    "primary_metric": "PR-AUC (average_precision)",
    "python": sys.version.split()[0], "sklearn": sklearn.__version__,
}
for k, v in log.items():
    print(f"{k:18s}: {v}")

---
## Sebelum Anda push

1. **Runtime → Run all.** Pastikan seluruh notebook jalan dari atas ke bawah tanpa error, dan outputnya tersimpan.
2. Pastikan setiap blok sudah Anda beri satu kalimat interpretasi.
3. Lengkapi `laporan/LAPORAN.md`:
   - minimal **5 temuan EDA** berformat **Temuan → Bukti → Implikasi**;
   - **tabel keputusan preprocessing** untuk dataset penelitian Anda sendiri;
   - **satu paragraf audit leakage** pada dataset penelitian Anda.
4. Isi identitas di sel paling atas notebook ini dan di `README.md`.
5. Simpan: **File → Save a copy in GitHub**.

Setelah push, GitHub Actions akan menjalankan pemeriksaan kelengkapan. Hasilnya muncul di tab **Actions** repositori Anda. Pemeriksaan itu bukan nilai — ia hanya memastikan pekerjaan Anda lengkap dan tidak melanggar aturan validitas paling dasar.

---

## Lima kalimat yang dibawa pulang

1. EDA adalah proses mengumpulkan bukti, bukan membuat grafik.
2. Setiap keputusan preprocessing harus dapat ditunjuk buktinya.
3. Split sebelum preprocessing yang belajar dari data; test set dikunci.
4. Accuracy pada data tidak seimbang menyesatkan ke dua arah — bandingkan dengan baseline.
5. Nilai terlalu sempurna adalah gejala, bukan prestasi.